In [17]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [18]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [19]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [20]:
#dataset = generate_dataset()

#with open("prompt-evals-dataset.json", "w") as f:
#    json.dump(dataset, f, indent=2)


In [21]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [22]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [23]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [24]:
with open("prompt-evals-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


In [25]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extraction from ARN\n\nHere's a comprehensive solution with multiple approaches:\n\n## Simple Solution\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str | None:\n    \"\"\"\n    Extract AWS account ID from an ARN string.\n    \n    Args:\n        arn: AWS ARN string (e.g., 'arn:aws:iam::123456789012:role/MyRole')\n    \n    Returns:\n        Account ID as string, or None if not present/applicable\n        \n    Examples:\n        >>> extract_account_id_from_arn('arn:aws:iam::123456789012:role/MyRole')\n        '123456789012'\n        >>> extract_account_id_from_arn('arn:aws:s3:::my-bucket')\n        None\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    parts = arn.split(':')\n    \n    # ARN format: arn:partition:service:region:account-id:resource\n    # Index 4 is the account ID\n    if len(parts) >= 5:\n        account_id = parts[4]\n        # Return account ID only if it's not empty (some serv